In [ ]:
from askslm import metadata, taxonomy
from askslm.contracts import FilterCriteria, TaxaSelector, SampleSelector
from Engines.MetadataEngine.metadata import Metadata

# Metadata
metadata_df = metadata.extract("/Users/yarintamam/Code/Ask-SLM/Workspaces/ABX/inputs/metadata.csv")
overview = metadata.overview(metadata_df)
overview.write("out/metadata_overview")

# Taxonomy: load → resolve → build abundance matrix
taxonomy_df = taxonomy.load("/Users/yarintamam/Code/Ask-SLM/Workspaces/ABX/inputs/taxonomy.csv")
resolved = taxonomy.resolve(taxonomy_df)
builder = taxonomy.AbundanceMatrixBuilder()
matrix = builder.build(builder.build(resolved.df), "genus")
matrix.write("out/abundance.tsv", fmt="maaslin")

# Cohort: Bacteria clade, WGS assay, exclude control subjects
CONTROL_SUBJECTS = ("CAN", "CAC", "CAM", "CAK", "CAA")

cohort = taxonomy.build_cohort(
    matrix,
    Metadata(df=metadata_df),
).filter(
    FilterCriteria(
        taxa=TaxaSelector(rank="superkingdom", names=("Bacteria",)),
        sample_filters=(
            SampleSelector(column="assay_type", values=("WGS",)),
            SampleSelector(column="host_subject_id_sam", values=CONTROL_SUBJECTS, exclude=True),
        ),
    )
)

cohort.write("out/amplicon_and_control_excluded_subset")

print("Cohort profile shape:", cohort.matrix.matrix.shape)
print("Control subjects in cohort:", sorted(cohort.metadata.df["host_subject_id_sam"].unique()))
print("Assay types in cohort:", cohort.metadata.df["assay_type"].unique().tolist())